# LLM Inference Benchmark - Quick Start

This notebook demonstrates the basic usage of the LLM inference benchmarking suite.

In [ ]:
# Install dependencies (if needed)
# !pip install -e ".[all]"

In [ ]:
import sys
sys.path.insert(0, '..')

from benchmark.runner import BenchmarkRunner, BenchmarkConfig
from benchmark.workloads import WorkloadGenerator
from strategies.base import BaselineStrategy
from serving.hf_backend import HuggingFaceBackend

## 1. Configure the Benchmark

In [ ]:
# Create benchmark configuration
config = BenchmarkConfig(
    warmup_runs=2,
    measurement_runs=5,
    output_dir='../results/quickstart',
    random_seed=42,
)

print(f"Warmup runs: {config.warmup_runs}")
print(f"Measurement runs: {config.measurement_runs}")

## 2. Create Components

In [ ]:
# Create strategy (baseline - no optimizations)
strategy = BaselineStrategy()
print(f"Strategy: {strategy.name}")
print(f"Description: {strategy.describe()}")

In [ ]:
# Create workload
workload_gen = WorkloadGenerator(seed=42)
workload = workload_gen.create('short', n=10)
print(f"Workload: {workload.name}")
print(f"Description: {workload.describe()}")

In [ ]:
# Create backend
backend = HuggingFaceBackend()
print(f"Backend: {backend.name}")

## 3. Run Benchmark

In [ ]:
# Create runner
runner = BenchmarkRunner(config=config)

# Run single benchmark (using a small model for demo)
model_name = 'EleutherAI/pythia-70m'

print(f"Running benchmark with {model_name}...")
result = runner.run_single(
    strategy=strategy,
    model=model_name,
    workload=workload,
    backend=backend,
)

## 4. Analyze Results

In [ ]:
# Print results
print("=" * 50)
print("BENCHMARK RESULTS")
print("=" * 50)
print(f"Strategy: {result.strategy}")
print(f"Model: {result.model}")
print(f"Backend: {result.backend}")
print()
print("Latency Metrics:")
for k, v in result.latency_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
print()
print("Throughput Metrics:")
for k, v in result.throughput_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
print()
print("Memory Metrics:")
for k, v in result.memory_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Save results
runner.results = [result]
results_path = runner.save_results('quickstart_results.json')
print(f"Results saved to: {results_path}")

## 5. Generate Report

In [ ]:
from analysis.report import ReportGenerator

report_gen = ReportGenerator(output_dir='../results/quickstart')
md_report = report_gen.generate_markdown([result.to_dict()])
print(md_report)